# Person Random Walk Publisher

This notebook creates a named person who performs a random walk starting from City Hall.
Location updates (with color) are published to MQTT once per second.

**Weather-aware:** The person seeks shelter at a coffee shop when it rains!

**Usage:**
1. Start `weather_service.ipynb` to broadcast weather cycles
2. Edit the `PERSON_NAME` and `COLOR` below
3. Run all cells to start the walker
4. Launch multiple copies of this notebook with different names to see multiple walkers
5. Use `map_viewer.ipynb` to visualize all walkers on a map

In [8]:
import asyncio
import json
import math
import random
import time

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

In [9]:
# ===== CONFIGURE YOUR PERSON HERE =====
PERSON_NAME = "Alice"  # Change this for each notebook!
COLOR = "#e74c3c"       # Hex color code (red). Change for different colors.
# ======================================

In [10]:
# City Hall coordinates (Copenhagen)
CITY_HALL_LNGLAT = (12.5683, 55.6761)

# Coffee shop shelter location (where persons go when it rains)
COFFEE_SHOP_LNGLAT = (12.5678, 55.6765)

# Random walk parameters
STEP_M = 6.0              # meters per step
STEP_S = 1.0              # seconds between steps
MAX_RADIUS_M = 250.0      # max distance from City Hall
SEED = int(time.time())   # random seed based on current time

# MQTT setup: connect to primary broker from config
# To use a different broker, edit config.yaml and change active_profiles
cfg = load_config()
connector = MqttConnector(cfg.mqtt, client_id_suffix=f"walker-{PERSON_NAME}")
connector.connect()
if not connector.wait_for_connection(timeout=10.0):
    raise RuntimeError("Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)
print(f"✓ Connected to MQTT broker as {PERSON_NAME}")
print(f"  Broker: {cfg.mqtt.host}:{cfg.mqtt.port}")

✓ Connected to MQTT broker as Alice
  Broker: 127.0.0.1:1883


In [11]:
# Weather subscription: React to rain/sunny weather
# Global flag to track current weather state
is_raining = False

def on_weather_message(client, userdata, message):
    """
    Callback when weather update arrives.
    
    Expected message format: {"raining": bool, "timestamp": float}
    """
    global is_raining
    try:
        data = json.loads(message.payload.decode())
        new_state = data["raining"]
        
        if new_state and not is_raining:
            print(f"☔ Weather changed: RAIN STARTED - {PERSON_NAME} seeks shelter!")
            is_raining = True
        elif not new_state and is_raining:
            print(f"☀️  Weather changed: RAIN STOPPED - {PERSON_NAME} goes outside!")
            is_raining = False
    except Exception as e:
        print(f"Error processing weather message: {e}")

# Subscribe to weather updates
connector.client.on_message = on_weather_message
connector.client.subscribe("weather/rain", qos=0)

print("✓ Subscribed to weather updates (weather/rain)")

✓ Subscribed to weather updates (weather/rain)


In [12]:
async def random_walk_publisher(
    person_name: str,
    color: str,
    *,
    seed: int = 42,
    step_m: float = 6.0,
    step_s: float = 1.0,
    max_radius_m: float = 250.0,
) -> None:
    """
    Perform a random walk and publish location updates to MQTT.
    
    When it rains, person seeks shelter at coffee shop.
    When it's sunny, person does random walk.
    
    Publishes to topic: persons/{person_name}/location
    Message format: {"lng": float, "lat": float, "color": str, "name": str, "timestamp": float}
    """
    rng = random.Random(seed)
    center_lng, center_lat = CITY_HALL_LNGLAT
    shelter_lng, shelter_lat = COFFEE_SHOP_LNGLAT
    meters_per_deg_lat = 111_320.0
    meters_per_deg_lng = 111_320.0 * math.cos(math.radians(center_lat))
    
    # Start at City Hall
    x_m = 0.0
    y_m = 0.0
    
    topic = f"persons/{person_name}/location"
    print(f"Starting random walk for {person_name} (color: {color})")
    print(f"Publishing to topic: {topic}")
    print(f"Press the stop button to stop.\n")
    
    step_count = 0
    while True:
        # Check weather and decide behavior
        if is_raining:
            # Seek shelter at coffee shop - stay still
            lng = shelter_lng
            lat = shelter_lat
        else:
            # Random walk when sunny
            theta = rng.random() * 2.0 * math.pi
            x_m += step_m * math.cos(theta)
            y_m += step_m * math.sin(theta)
            
            # Keep within max radius (soft boundary)
            r = math.hypot(x_m, y_m)
            if r > max_radius_m:
                scale = max_radius_m / r
                x_m *= scale
                y_m *= scale
            
            # Convert to lng/lat
            lng = center_lng + (x_m / meters_per_deg_lng)
            lat = center_lat + (y_m / meters_per_deg_lat)
        
        # Create and publish message (always publish, even when sheltering)
        message = {
            "lng": lng,
            "lat": lat,
            "color": color,
            "name": person_name,
            "timestamp": time.time(),
        }
        
        publisher.publish_json(topic, json.dumps(message), qos=0)
        
        step_count += 1
        if step_count % 10 == 0:
            status = "☔ sheltering" if is_raining else "☀️  walking"
            print(f"  {person_name}: {step_count} steps, {status}, at ({lng:.6f}, {lat:.6f})")
        
        await asyncio.sleep(step_s)

In [ ]:
# Start the random walk in the background
task = asyncio.create_task(
    random_walk_publisher(
        PERSON_NAME,
        COLOR,
        seed=SEED,
        step_m=STEP_M,
        step_s=STEP_S,
        max_radius_m=MAX_RADIUS_M,
    )
)
print(f"\n✓ {PERSON_NAME} is walking and publishing to MQTT!")
print("Run the next cell to stop.")


✓ Alice is walking and publishing to MQTT!
Run the next cell to stop.


Starting random walk for Alice (color: #e74c3c)
Publishing to topic: persons/Alice/location
Press the stop button to stop.

  Alice: 10 steps, ☀️  walking, at (12.568424, 55.676064)
  Alice: 20 steps, ☀️  walking, at (12.568285, 55.676084)
  Alice: 30 steps, ☀️  walking, at (12.568101, 55.676262)
  Alice: 40 steps, ☀️  walking, at (12.568298, 55.676179)
  Alice: 50 steps, ☀️  walking, at (12.568467, 55.676182)
  Alice: 60 steps, ☀️  walking, at (12.568478, 55.676160)
  Alice: 70 steps, ☀️  walking, at (12.568337, 55.676179)
  Alice: 80 steps, ☀️  walking, at (12.568466, 55.676274)
  Alice: 90 steps, ☀️  walking, at (12.568573, 55.676387)


In [14]:
# Stop the walker
task.cancel()
connector.disconnect()
print(f"✓ {PERSON_NAME} stopped walking.")

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


✓ Alice stopped walking.
